#  Adult Income – Feature Engineering Pipeline (Leak-Free)

Pipeline được chia thành 6 bước:
1. **Load Data** – Tải dữ liệu và mapping
2. **Transform** – Yeo-Johnson & Robust Scaler
3. **Train/Test Split**
4. **Encode** – Categorical encoding
5. **Interactions** – Econometric & Fairness features
6. **Save** – Lưu kết quả

---
##  Import thư viện

In [56]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer, RobustScaler
from sklearn.model_selection import train_test_split
from category_encoders import CatBoostEncoder, LeaveOneOutEncoder

warnings.filterwarnings('ignore')

# ── Đường dẫn ──────────────────────────────────────────────
BASE_DIR    = r"C:\Users\lanph\OneDrive\Desktop\Introduction to Data Science\Final_Project\Adult_Project_Final_Term"
DATA_IN     = os.path.join(BASE_DIR, "data", "processed", "adult_after_eda.csv")
MAPPING_CSV = os.path.join(BASE_DIR, "data", "processed", "mapping_.csv")
PROCESSED   = os.path.join(BASE_DIR, "data", "processed")
ENCODER_DIR = os.path.join(BASE_DIR, "models", "encoders")
os.makedirs(ENCODER_DIR, exist_ok=True)
os.makedirs(PROCESSED,   exist_ok=True)


---
##   Load Data

In [57]:
df = pd.read_csv(DATA_IN)
print(f'    Shape: {df.shape}')  # expected (48842, 13)
df.head(3)

    Shape: (48842, 13)


,age,workclass,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K


In [58]:
# Load country → income group mapping
mapping_df = pd.read_csv(MAPPING_CSV)
country_map = dict(zip(mapping_df.iloc[:, 0], mapping_df.iloc[:, 1]))
print(f'[CountryIncomeGroup] Loaded {len(country_map)} country mappings')
mapping_df.head(5)

[CountryIncomeGroup] Loaded 42 country mappings


,country_adult_dataset,income_group
0,United-States,H
1,Cuba,LM
2,Jamaica,LM
3,India,L
4,?,UM


---
## Transform: Yeo-Johnson & Robust Scaler

>  **Leak-free**: Fit chỉ trên **toàn bộ dữ liệu trước split** với Yeo-Johnson (không dùng target).  
> Robust Scaler sẽ được fit lại trên **train only** sau split.

In [59]:
print('[2a] Yeo-Johnson fit …')

YJ_COLS = ['capital-gain', 'capital-loss']

yj = PowerTransformer(method='yeo-johnson', standardize=False)
df[YJ_COLS] = yj.fit_transform(df[YJ_COLS])

print(f'[YeoJohnson] Fitted on columns: {YJ_COLS}')
for col, lam in zip(YJ_COLS, yj.lambdas_):
    print(f'  λ({col}) = {lam:.4f}')

# Lưu transformer
yj_path = os.path.join(ENCODER_DIR, 'yeo_johnson.pkl')
joblib.dump(yj, yj_path)
print(f'[YeoJohnson] Transformer saved → {yj_path}')

[2a] Yeo-Johnson fit …
[YeoJohnson] Fitted on columns: ['capital-gain', 'capital-loss']
  λ(capital-gain) = -1.3724
  λ(capital-loss) = -2.8492
[YeoJohnson] Transformer saved → C:\Users\lanph\OneDrive\Desktop\Introduction to Data Science\Final_Project\Adult_Project_Final_Term\models\encoders\yeo_johnson.pkl


---
##  Train / Test Split (80% / 20%)

In [60]:
print('[3] Train/test split (80% / 20%) …')

# Encode target safely for both raw-string and already-numeric income values
income_series = df['income']
if pd.api.types.is_numeric_dtype(income_series):
    df['income'] = income_series.astype(int)
else:
    df['income'] = (
        income_series.astype(str).str.strip().isin(['>50K', '>50K.'])
    ).astype(int)

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['income']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'    Train: {len(train_df):,}  |  Test: {len(test_df):,}')
print(f'    >50K ratio — Train: {train_df["income"].mean():.3f}  Test: {test_df["income"].mean():.3f}')

[3] Train/test split (80% / 20%) …
    Train: 39,073  |  Test: 9,769
    >50K ratio — Train: 0.239  Test: 0.239


In [61]:
print('[2b] Robust Scaler fit (on TRAIN only) …')

RS_COLS = ['age', 'hours-per-week', 'education-num']

rs = RobustScaler()
train_df[RS_COLS] = rs.fit_transform(train_df[RS_COLS])
test_df[RS_COLS]  = rs.transform(test_df[RS_COLS])

print(f'[RobustScaler] Fitted on columns: {RS_COLS}')
for col, med, iqr in zip(RS_COLS, rs.center_, rs.scale_):
    print(f'  {col}: median={med:.2f}, IQR={iqr:.2f}')

rs_path = os.path.join(ENCODER_DIR, 'robust_scaler.pkl')
joblib.dump(rs, rs_path)
print(f'[RobustScaler] Scaler saved → {rs_path}')

[2b] Robust Scaler fit (on TRAIN only) …
[RobustScaler] Fitted on columns: ['age', 'hours-per-week', 'education-num']
  age: median=37.00, IQR=20.00
  hours-per-week: median=40.00, IQR=5.00
  education-num: median=10.00, IQR=3.00
[RobustScaler] Scaler saved → C:\Users\lanph\OneDrive\Desktop\Introduction to Data Science\Final_Project\Adult_Project_Final_Term\models\encoders\robust_scaler.pkl


---
##  Encode: Categorical Variables

| Biến | Phương pháp | Ghi chú |
|---|---|---|
| `native-country` | Ordinal (0–3) | Theo nhóm thu nhập quốc gia |
| `occupation` | Group mapping | 4 nhóm: manual, high_skill, office_tech, specialised |
| `marital-status` | Binary flag | married_flag |
| `relationship` | One-Hot | |
| `race` | One-Hot | 5 cột |
| `sex` | Binary | Male=1 |
| `occupation_group` | CatBoost Encoder | Fit train only |
| `workclass` | LOO Encoder | Fit train only |

In [62]:
def encode_non_target(df_in, fit=False,
                       country_map=None,
                       cb_enc=None, loo_enc=None,
                       target=None):
    """Apply all categorical encodings. fit=True for train, False for test."""
    df = df_in.copy()

    # ── Country Income Group (Ordinal) ──────────────────────
    country_group = df['native-country'].map(country_map)
    country_group_str = country_group.fillna('UNK').astype(str).str.strip().str.upper()

    numeric_group = pd.to_numeric(country_group_str, errors='coerce')
    if numeric_group.notna().all():
        df['country_income_group'] = numeric_group.astype(int)
    else:
        income_group_order = {'L': 0, 'LM': 1, 'UM': 2, 'H': 3, 'LOW': 0, 'MID': 1, 'MIDDLE': 1, 'UPPER-MIDDLE': 2, 'HIGH': 3, 'UNK': 0}
        mapped_group = country_group_str.map(income_group_order)
        df['country_income_group'] = mapped_group.fillna(0).astype(int)

    df.drop(columns=['native-country'], inplace=True)
    print(f'[CountryIncomeGroup] native-country → country_income_group')
    print(df['country_income_group'].value_counts().sort_index())

    # ── Occupation Group ────────────────────────────────────
    occ_map = {
        'Exec-managerial': 'high_skill',  'Prof-specialty': 'high_skill',
        'Tech-support':    'office_tech', 'Adm-clerical':   'office_tech',
        'Sales':           'office_tech', 'Protective-serv':'specialised',
        'Priv-house-serv': 'specialised', 'Armed-Forces':   'specialised',
        'Craft-repair':    'manual',      'Machine-op-inspct':'manual',
        'Transport-moving':'manual',      'Handlers-cleaners':'manual',
        'Farming-fishing': 'manual',      'Other-service':   'manual',
    }
    df['occupation_group'] = df['occupation'].map(occ_map).fillna('manual')
    df.drop(columns=['occupation'], inplace=True)
    print(f'[OccupationGroup] occupation → occupation_group')
    print(df['occupation_group'].value_counts())

    # ── Marital Status → Binary flag ────────────────────────
    married_vals = ['Married-civ-spouse', 'Married-AF-spouse']
    df['married_flag'] = df['marital-status'].isin(married_vals).astype(int)
    df.drop(columns=['marital-status'], inplace=True)
    pct = df['married_flag'].mean() * 100
    print(f'[MaritalStatus] married_flag: {df["married_flag"].sum()}/{len(df)} married ({pct:.1f}%)')

    # ── Relationship → One-Hot (drop_first=True → 5 cột thay vì 6) ──────────
    rel_dummies = pd.get_dummies(df['relationship'], prefix='relationship')
    df = pd.concat([df.drop(columns=['relationship']), rel_dummies], axis=1)
    print(f'[MaritalStatus] One-Hot encoded: relationship → {list(rel_dummies.columns)}')

    # ── Race → One-Hot ───────────────────────────────────────
    race_dummies = pd.get_dummies(df['race'], prefix='race')
    df = pd.concat([df.drop(columns=['race']), race_dummies], axis=1)
    print(f'[OtherEncoding] One-Hot encoded race → {list(race_dummies.columns)}')

    # ── Sex → Binary ─────────────────────────────────────────
    df['sex_binary'] = (df['sex'].astype(str).str.strip() == 'Male').astype(int)
    df.drop(columns=['sex'], inplace=True)
    print(f'[OtherEncoding] sex → sex_binary (Male=1, Female=0)')

    # ── Occupation Group → CatBoost Encoder ─────────────────
    if fit:
        cb_enc = CatBoostEncoder(cols=['occupation_group'], random_state=42)
        df['occupation_group'] = cb_enc.fit_transform(df['occupation_group'], target)
        joblib.dump(cb_enc, os.path.join(ENCODER_DIR, 'occupation_catboost.pkl'))
        print('[OccupationGroup] CatBoost encoder fitted & saved')
    else:
        df['occupation_group'] = cb_enc.transform(df[['occupation_group']])
        print('[OccupationGroup] CatBoost encoder applied (test)')

    # ── Workclass → LOO Encoder ──────────────────────────────
    if fit:
        loo_enc = LeaveOneOutEncoder(cols=['workclass'], random_state=42)
        df['workclass'] = loo_enc.fit_transform(df['workclass'], target)
        joblib.dump(loo_enc, os.path.join(ENCODER_DIR, 'workclass_loo.pkl'))
        print('[OtherEncoding] LOO encoder fitted & saved')
    else:
        df['workclass'] = loo_enc.transform(df[['workclass']])
        print('[OtherEncoding] LOO encoder applied (test)')

    return df, cb_enc, loo_enc

print(' encode_non_target() defined')

 encode_non_target() defined


In [63]:
print('[4] Non-target transforms — TRAIN …')
train_enc, cb_enc, loo_enc = encode_non_target(
    train_df, fit=True,
    country_map=country_map,
    target=train_df['income']
)

print('\n[4] Non-target transforms — TEST …')
test_enc, _, _ = encode_non_target(
    test_df, fit=False,
    country_map=country_map,
    cb_enc=cb_enc, loo_enc=loo_enc
)

[4] Non-target transforms — TRAIN …
[CountryIncomeGroup] native-country → country_income_group
country_income_group
0      436
1      997
2     1583
3    36057
Name: count, dtype: int64
[OccupationGroup] occupation → occupation_group
occupation_group
manual         17471
high_skill     10579
office_tech    10003
specialised     1020
Name: count, dtype: int64
[MaritalStatus] married_flag: 17979/39073 married (46.0%)
[MaritalStatus] One-Hot encoded: relationship → ['relationship_Husband', 'relationship_Not-in-family', 'relationship_Other-relative', 'relationship_Own-child', 'relationship_Unmarried', 'relationship_Wife']
[OtherEncoding] One-Hot encoded race → ['race_Amer-Indian-Eskimo', 'race_Asian-Pac-Islander', 'race_Black', 'race_Other', 'race_White']
[OtherEncoding] sex → sex_binary (Male=1, Female=0)
[OccupationGroup] CatBoost encoder fitted & saved
[OtherEncoding] LOO encoder fitted & saved

[4] Non-target transforms — TEST …
[CountryIncomeGroup] native-country → country_income_grou

---
##  Interaction Features: Econometric & Fairness

| Feature | Công thức | Ý nghĩa kinh tế |
|---|---|---|
| `human_capital` | age × education-num | Vốn con người |
| `household_labour` | hours-per-week × married_flag | Lao động hộ gia đình |
| `net_capital` | capital-gain − capital-loss | Thu nhập vốn ròng |
| `edu_x_[race]` | education-num × race_* | Fairness: Giáo dục theo chủng tộc |
| `hours_x_[race]` | hours-per-week × race_* | Fairness: Giờ làm theo chủng tộc |
| `capital_x_[race]` | net_capital × race_* | Fairness: Vốn theo chủng tộc |

In [64]:
RACE_COLS = ['Amer-Indian-Eskimo', 'Asian-Pac-Islander', 'Black', 'Other', 'White']

def add_interactions(df_in):
    df = df_in.copy()

    # ── Econometric ─────────────────────────────────────────
    df['human_capital']    = df['age'] * df['education-num']
    df['household_labour'] = df['hours-per-week'] * df['married_flag']
    df['net_capital']      = df['capital-gain'] - df['capital-loss']

    print(f'[HumanCapital]    human_capital    | mean={df["human_capital"].mean():.2f}, std={df["human_capital"].std():.2f}')
    print(f'[HouseholdLabour] household_labour | mean={df["household_labour"].mean():.2f}, std={df["household_labour"].std():.2f}')
    print(f'[NetCapital]      net_capital       | mean={df["net_capital"].mean():.4f}, std={df["net_capital"].std():.4f}')

    # ── Fairness Interactions ───────────────────────────────
    edu_cols     = []
    hours_cols   = []
    capital_cols = []

    for race in RACE_COLS:
        rc = f'race_{race}'
        if rc in df.columns:
            df[f'edu_x_{race}']     = df['education-num']  * df[rc]
            df[f'hours_x_{race}']   = df['hours-per-week'] * df[rc]
            df[f'capital_x_{race}'] = df['net_capital']    * df[rc]
            edu_cols.append(f'edu_x_{race}')
            hours_cols.append(f'hours_x_{race}')
            capital_cols.append(f'capital_x_{race}')

    print(f'[EduByRace]     Created {len(edu_cols)} interaction columns: {edu_cols}')
    print(f'[HoursByRace]   Created {len(hours_cols)} interaction columns: {hours_cols}')
    print(f'[CapitalByRace] Created {len(capital_cols)} interaction columns: {capital_cols}')

    return df


In [65]:
print('[6] Econometric & fairness interactions …')

train_final = add_interactions(train_enc)
test_final  = add_interactions(test_enc)

# Attach target (income đã được encode ở bước split)
train_final['income'] = train_df['income'].values
test_final['income']  = test_df['income'].values

print(f'\n    [TRAIN] shape: {train_final.shape}')
print(f'    [TEST]  shape: {test_final.shape}')

[6] Econometric & fairness interactions …
[HumanCapital]    human_capital    | mean=0.02, std=0.66
[HouseholdLabour] household_labour | mean=0.31, std=1.67
[NetCapital]      net_capital       | mean=0.0432, std=0.2173
[EduByRace]     Created 5 interaction columns: ['edu_x_Amer-Indian-Eskimo', 'edu_x_Asian-Pac-Islander', 'edu_x_Black', 'edu_x_Other', 'edu_x_White']
[HoursByRace]   Created 5 interaction columns: ['hours_x_Amer-Indian-Eskimo', 'hours_x_Asian-Pac-Islander', 'hours_x_Black', 'hours_x_Other', 'hours_x_White']
[CapitalByRace] Created 5 interaction columns: ['capital_x_Amer-Indian-Eskimo', 'capital_x_Asian-Pac-Islander', 'capital_x_Black', 'capital_x_Other', 'capital_x_White']
[HumanCapital]    human_capital    | mean=0.03, std=0.66
[HouseholdLabour] household_labour | mean=0.28, std=1.65
[NetCapital]      net_capital       | mean=0.0462, std=0.2229
[EduByRace]     Created 5 interaction columns: ['edu_x_Amer-Indian-Eskimo', 'edu_x_Asian-Pac-Islander', 'edu_x_Black', 'edu_x_Oth

---
##  Save Processed Data

In [66]:
print('[7] Attaching target & saving …')

train_out_path = os.path.join(PROCESSED, 'adult_features_train.csv')
test_out_path  = os.path.join(PROCESSED, 'adult_features_test.csv')

train_final.to_csv(train_out_path, index=False)
test_final.to_csv(test_out_path,  index=False)

[7] Attaching target & saving …
